# Dataset generation + first-4-message stats

Новый ноутбук на базе `dataset_generation.ipynb`.
Добавлена статистика по первым 4 сообщениям в диалоге: `user -> tool call -> tool response -> assistant`.

In [1]:
import json
import numpy as np
import pandas as pd

np.random.seed(1241)


In [2]:
init_dataset = pd.read_json('hf://datasets/Team-ACE/ToolACE/data.json')
print(f'Rows: {len(init_dataset)}')
init_dataset.head(2)


/Users/valeria/Desktop/skoltech/DL/dl_project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows: 11300


,system,conversations
0,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'I'm considering in..."
1,You are an expert in composing functions. You ...,"[{'from': 'user', 'value': 'Could you please f..."


In [3]:
# Как выглядит один диалог
init_dataset['conversations'][0]


[{'from': 'user',
  'value': "I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?"},
 {'from': 'assistant',
  'value': '[Market Trends API(trend_type="MARKET_INDEXES", country="us")]'},
 {'from': 'tool',
  'value': '[{"name": "Market Trends API", "results": {"trends": [{"name": "S&P 500", "description": "Standard & Poor\'s 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data": {"current_value": "4172.80", "percentage_change": "+0.68%"}}, {"name": "DOW J", "description": "Dow Jones Industrial Average is a price-weighted average of 30 blue-chip stocks that are generally the leaders in their industry.", "data": {"current_value": "34479.60", "percentage_change": "+0.47%"}}, {"name": "NASDAQ", "description": "The NASDAQ Composite is a broad-based capitalization-weighted index of stocks in all three NASDAQ tiers: Global Select, Global Market and Cap

In [4]:
# Системный промпт с описанием тулинга
init_dataset['system'][0]


'You are an expert in composing functions. You are given a question and a set of possible functions. \nBased on the question, you will need to make one or more function/tool calls to achieve the purpose. \nIf none of the function can be used, point it out. If the given question lacks the parameters required by the function,\nalso point it out. You should only return the function call in tools call sections.\nHere is a list of functions in JSON format that you can invoke:\n[{"name": "newAddress", "description": "Generates a new Ethereum address that can be used to send or receive funds. Do not lose the password! We can\'t restore access to an address if you lose it.", "parameters": {"type": "dict", "properties": {"password": {"description": "The password for the new Ethereum address", "type": "string"}}, "required": ["password"]}, "required": null}, {"name": "Market Trends API", "description": "Get the latest market trends and relevant news for a specified country and language.", "param

In [5]:
def is_tool_call_message(message):
    """Heuristic for assistant tool-call messages in ToolACE format."""
    if message.get('from') != 'assistant':
        return False

    value = message.get('value', '')
    if not isinstance(value, str):
        return False

    value = value.strip()
    if not value:
        return False

    # 1) JSON-style tool call: [{"name": ...}] or {"name": ...}
    if (value.startswith('[') and value.endswith(']')) or (value.startswith('{') and value.endswith('}')):
        try:
            obj = json.loads(value)
            if isinstance(obj, list) and len(obj) > 0:
                if all(isinstance(x, dict) and ('name' in x or 'function' in x) for x in obj):
                    return True
            if isinstance(obj, dict) and ('name' in obj or 'function' in obj):
                return True
        except Exception:
            pass

    # 2) ToolACE string-style tool call: [ToolName(arg="...")]
    #    Important: normal assistant answers almost never start with '[' and end with ']'
    #    and include call-like parentheses.
    if value.startswith('[') and value.endswith(']'):
        inner = value[1:-1].strip()
        if '(' in inner and ')' in inner:
            return True

    return False


def classify_message(message):
    role = message.get('from')
    if role == 'user':
        return 'user'
    if role == 'tool':
        return 'tool_response'
    if role == 'assistant':
        if is_tool_call_message(message):
            return 'tool_call'
        return 'assistant'
    return role or 'unknown'


def first_four_types(conv):
    first_four = conv[:4]
    return [classify_message(m) for m in first_four]



In [6]:
# Статистика по первым четырем сообщениям
rows = []
for i, conv in enumerate(init_dataset['conversations']):
    types = first_four_types(conv)
    rows.append({
        'idx': i,
        'len_conv': len(conv),
        'first4_types': types,
        'first4_signature': ' -> '.join(types),
        'has_4_messages': len(conv) >= 4,
        'matches_target_pattern': len(types) == 4 and types == ['user', 'tool_call', 'tool_response', 'assistant']
    })

first4_df = pd.DataFrame(rows)
first4_df.head()


,idx,len_conv,first4_types,first4_signature,has_4_messages,matches_target_pattern
0,0,10,"[user, tool_call, tool_response, assistant]",user -> tool_call -> tool_response -> assistant,True,True
1,1,10,"[user, tool_call, tool_response, assistant]",user -> tool_call -> tool_response -> assistant,True,True
2,2,10,"[user, tool_call, tool_response, assistant]",user -> tool_call -> tool_response -> assistant,True,True
3,3,12,"[user, tool_call, tool_response, assistant]",user -> tool_call -> tool_response -> assistant,True,True
4,4,10,"[user, tool_call, tool_response, assistant]",user -> tool_call -> tool_response -> assistant,True,True


In [7]:
summary = {
    'total_dialogs': int(len(first4_df)),
    'dialogs_with_at_least_4_messages': int(first4_df['has_4_messages'].sum()),
    'matches_user_toolcall_toolresponse_assistant': int(first4_df['matches_target_pattern'].sum()),
}

summary['match_rate_among_all'] = round(summary['matches_user_toolcall_toolresponse_assistant'] / summary['total_dialogs'], 4)
summary['match_rate_among_len_ge_4'] = round(
    summary['matches_user_toolcall_toolresponse_assistant'] / max(summary['dialogs_with_at_least_4_messages'], 1),
    4,
)

pd.Series(summary)


total_dialogs                                   11300.0000
dialogs_with_at_least_4_messages                  800.0000
matches_user_toolcall_toolresponse_assistant      442.0000
match_rate_among_all                                0.0391
match_rate_among_len_ge_4                           0.5525
dtype: float64

In [8]:
# Топ самых частых шаблонов первых 4 сообщений
first4_df['first4_signature'].value_counts().head(15).to_frame('count')


,count
first4_signature,
user -> tool_call,8493
user -> assistant,2007
user -> tool_call -> tool_response -> assistant,442
user -> tool_call -> tool_response -> tool_call,292
user -> assistant -> user -> tool_call,60
user -> assistant -> user -> assistant,6


In [9]:
# Примеры диалогов, где паттерн совпадает
matched_indices = first4_df.loc[first4_df['matches_target_pattern'], 'idx'].head(10).tolist()
matched_indices


[0, 1, 2, 3, 4, 5, 6, 7, 8, 10]

In [10]:
# Посмотреть первые 4 сообщения для примеров
for idx in matched_indices:
    print(f'\n=== Conversation #{idx} ===')
    conv = init_dataset['conversations'][idx]
    for m in conv[:4]:
        msg_type = classify_message(m)
        text = str(m.get('value', ''))
        text = text[:220].replace('\n', ' ')
        print(f"{m.get('from'):>9} | {msg_type:>13} | {text}")



=== Conversation #0 ===
     user |          user | I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?
assistant |     tool_call | [Market Trends API(trend_type="MARKET_INDEXES", country="us")]
     tool | tool_response | [{"name": "Market Trends API", "results": {"trends": [{"name": "S&P 500", "description": "Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.", "data":
assistant |     assistant | Here are the top Market Trends in the US right now:  1. **S&P 500**: The Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies. Its current value is 4172.

=== Conversation #1 ===
     user |          user | Could you please find me some quotes about "inspiration"?
assistant |     tool_call | [Quotes by Keywords(word="inspiration")]
     tool | tool_response | [{"name": "

In [11]:
target = 'user -> assistant'
first4_df.loc[first4_df['first4_signature'] == target, ['idx', 'len_conv']].head(10)

,idx,len_conv
800,800,2
801,801,2
802,802,2
803,803,2
804,804,2
806,806,2
807,807,2
808,808,2
809,809,2
810,810,2


In [12]:
i = 800 # нужный idx
init_dataset['conversations'][i][:4]

[{'from': 'user',
  'value': 'I\'m planning some investments and need a comprehensive update. Could you provide the latest trading data for the NASDAQ: AAPL on a daily interval? Also, retrieve the latest real-time quote and detailed insider information for AAPL. Additionally, validate this IBAN number "DE64123905123140219513" for accuracy, and check the details of this Brazilian boleto with the line "23793.38129 60007.535305 22007.143305 8 84670000012345".'},
 {'from': 'assistant',
  'value': 'None of the functions provided can be used to achieve the stated requirements. The question needs specific trading data, real-time quotes, insider information, and validation of an IBAN number and Brazilian boleto, which are not covered by the available functions.'}]

In [13]:
first4_df.groupby('len_conv').size().to_frame('count').reset_index().sort_values('len_conv')

,len_conv,count
0,2,10500
1,4,6
2,6,262
3,8,232
4,10,207
5,12,93


In [14]:
def extract_tools_list_from_system(system_text: str):
    """
    Extract the *outer* JSON list of tool dicts from a ToolACE system prompt.

    Works even when the JSON contains nested [...] like enums/required, because it
    balances brackets and ignores brackets inside strings.
    """
    # Optional anchor: start searching after the "Here is a list..." line if present
    anchor = "Here is a list of functions"
    start_pos = system_text.find(anchor)
    if start_pos != -1:
        s = system_text[start_pos:]
    else:
        s = system_text

    # Find first '[' (start of the tool list)
    i0 = s.find("[")
    if i0 == -1:
        return None

    depth = 0
    in_str = False
    quote = ""
    esc = False
    start = None

    for i in range(i0, len(s)):
        ch = s[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
            continue

        if ch in ('"', "'"):
            in_str = True
            quote = ch
            continue

        if ch == "[":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "]":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    block = s[start:i + 1]
                    try:
                        obj = json.loads(block)
                    except Exception:
                        return None

                    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj) and any("name" in x for x in obj):
                        return obj
                    return None

    return None


In [15]:
# Уникальные названия тулов из всего датасета
all_tool_names = set()

for system_text in init_dataset['system']:
    tools = extract_tools_list_from_system(system_text)
    if tools:
        for tool in tools:
            if 'name' in tool:
                all_tool_names.add(tool['name'])

all_tool_names = sorted(all_tool_names)

print(f'Number of unique tools: {len(all_tool_names)}')
all_tool_names[:50]

Number of unique tools: 16134


['/1.3/add_working_days',
 '/1.3/list_non_working_days',
 '/BacterialMeningitisScoreForChildren',
 '/Calculate_expenses',
 '/Events/GetBookableItemList',
 '/GET_TAekwondo_Athlete_Ranking',
 '/GET_U-58_ATHLETE_RANKING',
 '/GetArchiveReport',
 '/GetUKRail',
 '/GetUpstrings',
 '/PaymentCalculatorSimple/Calculate',
 '/SearchPlayer',
 '/SetTag',
 '/addresses',
 '/addresses/autocomplete',
 '/api/TripDetails/{MFRef}',
 '/api/countriesAvailableToShipping',
 '/api/currenciesAvailable',
 '/api/v1/beyblades/{id}',
 '/api/v1/sources',
 '/batch',
 '/cinemas/{id}/films',
 '/companies/company_data',
 '/companies/{id}/events',
 '/crypto/intraday',
 '/dad-jokes/random',
 '/dad-jokes/search',
 '/detect',
 '/domain_check',
 '/echo',
 '/eff-wordlist',
 '/email-validator/health',
 '/email-validator/valistring',
 '/email/exist',
 '/email/format',
 '/email/valistring',
 '/equity/dividends',
 '/equity/earnings',
 '/equity/financial',
 '/events/{eventId}/overview',
 '/events/{eventId}/parameters',
 '/extract',

In [16]:
called_tool_names = set()

for conv in init_dataset['conversations']:
    for message in conv:
        if message.get('from') == 'tool':
            try:
                tool_results = json.loads(message['value'])
                for item in tool_results:
                    if 'name' in item:
                        called_tool_names.add(item['name'])
            except Exception:
                pass

called_tool_names = sorted(called_tool_names)

print(f'Number of actually called tools: {len(called_tool_names)}')
called_tool_names[:50]

Number of actually called tools: 1304


['/addresses',
 '/email/valistring',
 '/extract',
 '/forex/signal',
 '/madlibs-diceware',
 '/movies',
 '/playlist/info',
 '/stickers/trending',
 '1 Hour / Minutely Forecast (Nowcast)',
 '3dsMax_create_model',
 '3dsMax_render_scene',
 '567 Live App 2022',
 'AI BOT',
 'AUTOComplete',
 'Abuse Contact Lookup',
 'AdaptationPlanningService',
 'AgileResourceAllocator.allocateTeam',
 'AgileTaskScheduler.scheduleSprint',
 'AiNameComplete',
 'Aircraft Scatter Data',
 'Airplane Search',
 'All Rates',
 'Analyze V2',
 'AnalyzeGeneticDivergence.computeDivergence',
 'Angular Jobs API',
 'Article Extraction API',
 'ArtifactTimelineGenerator.generateTimeline',
 'Auto Complete API',
 'Autocomplete API',
 'Autocomplete Localities',
 'Autocomplete US Cities',
 'Available Symbols Endpoint',
 'BART Advisory Information',
 'BackupScheduler.scheduleBackup',
 'BacterialCultureAnalyzer',
 'BankPerformanceAnalysis.retrieveQuarterlyReport',
 'Basketball Live Matches API',
 'Billboard 200',
 'Billboard 200 Albums 

In [24]:
tool_name = "/email/valistring"

system_matches = init_dataset[
    init_dataset['system'].str.contains(tool_name, case=False, na=False)
].copy()

print(f'Found {len(system_matches)} system prompts with tool: {tool_name}')
system_matches[['system']]['system'][19]

Found 3 system prompts with tool: /email/valistring


'You are an expert in composing functions. You are given a question and a set of possible functions. \nBased on the question, you will need to make one or more function/tool calls to achieve the purpose. \nIf none of the function can be used, point it out. If the given question lacks the parameters required by the function,\nalso point it out. You should only return the function call in tools call sections.\nHere is a list of functions in JSON format that you can invoke:\n[{"name": "Get Email Messages", "description": "Retrieve email messages from a temporary email account.", "parameters": {"type": "dict", "properties": {"email": {"description": "The email address of the temporary email account.", "type": "string"}}, "required": ["email"]}, "required": null}, {"name": "exportEntities", "description": "Export entities (e.g., emails, contacts) from the specified email domain.", "parameters": {"type": "dict", "properties": {"outputFormat": {"description": "The format of the exported data 

In [26]:
tool_name = "/email/valistring"

dialog_matches = []

for i, conv in enumerate(init_dataset['conversations']):
    found = False
    for message in conv:
        if message.get('from') == 'tool':
            try:
                tool_results = json.loads(message['value'])
                for item in tool_results:
                    if item.get('name') == tool_name:
                        found = True
                        break
            except Exception:
                pass
        if found:
            dialog_matches.append(i)
            break

print(f'Found {len(dialog_matches)} dialogs where tool was called: {tool_name}')
dialog_matches[:20]

Found 1 dialogs where tool was called: /email/valistring


[19]